In [ ]:
import os

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_poisson_deviance

import config
from fixture_loader import load_fixtures, both_prem, filter_prem, add_rest_days
from pipeline.context import load_pipeline_context
from pipeline.player_base import build_player_base, TRANSFERRED_OUT
from pipeline.form import attach_form, build_form, compute_form_score
from pipeline.team_strength import (
    load_elo_all, build_elo_long, attach_match_elo,
    build_team_table, add_diffs, print_team_strength_diagnostics,
)
from pipeline.predictions import build_train_test, fit_goal_models, predict_fixtures
from pipeline.team_scores import build_team_fixture_scores, attach_team_and_fixture_scores, to_friendly
from pipeline.last_season import attach_last_season_stats
from pipeline.scoring import (
    attach_minutes_factors, attach_expected_return_signals, assemble_board,
    scale_global, expected_fpl_p90, compute_vorp_dynamic,
)
from pipeline.draft_ownership import build_all_boards
from pipeline.board_output import enrich_and_export, print_player_debug

In [ ]:
%load_ext autoreload
%autoreload 2

ctx = load_pipeline_context()

# unpack into globals for ad hoc / exploratory cells
GW, LAST_GW, TOTAL_MINUTES = ctx.GW, ctx.LAST_GW, ctx.TOTAL_MINUTES
FPL_DATA_FOLDER, PREV_DATA_FOLDER = ctx.FPL_DATA_FOLDER, ctx.PREV_DATA_FOLDER
fixtures_dir, prev_dir = ctx.fixtures_dir, ctx.prev_dir
players, teams, prev_teams = ctx.players, ctx.teams, ctx.prev_teams
playerstats, last_year_stats = ctx.playerstats, ctx.last_year_stats
elo_hist, elo = ctx.elo_hist, ctx.elo

In [ ]:
print(teams[["code","short_name"]].sort_values(by="code"))

In [ ]:
#### PLAYER STATS BASE
merged, base = build_player_base(playerstats, players, teams, TRANSFERRED_OUT)

In [ ]:
### FORM SCORE
merged = attach_form(merged, GW, FPL_DATA_FOLDER)

In [ ]:
### FIXTURE ORGANIZATION
this_seasons_fixtures = load_fixtures(fixtures_dir)
prev_seasons_fixtures = load_fixtures(prev_dir, played_only=True)

## ISOLATING PREM
this_season_prem_matches = filter_prem(this_seasons_fixtures)
prev_fx = filter_prem(both_prem(prev_seasons_fixtures, prev_teams))

print(f"GW{GW} fixtures: {(this_season_prem_matches['gameweek'] == GW).sum()}")

In [ ]:
this_season_prem_matches = add_rest_days(this_season_prem_matches, this_seasons_fixtures)
prev_fx = add_rest_days(prev_fx, prev_seasons_fixtures)

print(f"GW{GW} fixtures: {(this_season_prem_matches['gameweek'] == GW).sum()}")

In [ ]:
# #### FIXTURE GAMBLER MACHINE LEARNING MODEL
elo_all = load_elo_all()
elo_long = build_elo_long(elo_all)
this_season_prem_matches = attach_match_elo(this_season_prem_matches, elo_long)

played, team_fixtures = build_team_table(this_season_prem_matches, GW)
prev_played, prev_team_fixtures = build_team_table(prev_fx, 38)

played      = add_diffs(played, team_fixtures)
prev_played = add_diffs(prev_played, prev_team_fixtures)

latest = print_team_strength_diagnostics(team_fixtures, teams, this_season_prem_matches)

In [ ]:
train, test, FEATURES = build_train_test(played, prev_played, GW)

# ---- exploratory: Poisson vs flat baseline on this-season holdout ----
tr = train[train["season"] == 0]
ho = train[train["season"] == 1]

def fit_pred(y_col):
    m = sm.GLM(tr[y_col], sm.add_constant(tr[FEATURES]),
               family=sm.families.Poisson()).fit()
    p = m.predict(sm.add_constant(ho[FEATURES], has_constant="add"))
    return m, p

m_h, p_h = fit_pred("home_non_penalty_xg")
m_a, p_a = fit_pred("away_non_penalty_xg")

flat_h = tr["home_non_penalty_xg"].mean()
flat_a = tr["away_non_penalty_xg"].mean()

print("home  poisson", mean_poisson_deviance(ho["home_non_penalty_xg"], p_h),
      "flat", mean_poisson_deviance(ho["home_non_penalty_xg"], [flat_h]*len(ho)))
print("away  poisson", mean_poisson_deviance(ho["away_non_penalty_xg"], p_a),
      "flat", mean_poisson_deviance(ho["away_non_penalty_xg"], [flat_a]*len(ho)))
print(m_h.summary().tables[1])

In [ ]:
# ---- exploratory: pooled model with is_home ----
def stack_home_and_away(match_frame):
    home_rows = pd.DataFrame({
        "team_xg": match_frame["home_non_penalty_xg"],
        "elo_context": match_frame["elo_context"],
        "is_home": 1,
    })
    away_rows = pd.DataFrame({
        "team_xg": match_frame["away_non_penalty_xg"],
        "elo_context": -match_frame["elo_context"],   # away team's perspective
        "is_home": 0,
    })
    return pd.concat([home_rows, away_rows], ignore_index=True)

stacked_last_season = stack_home_and_away(tr)
stacked_this_season = stack_home_and_away(ho)
print(stacked_last_season.groupby("is_home")["team_xg"].mean())
print(stacked_this_season.groupby("is_home")["team_xg"].mean())

pooled_features = ["elo_context", "is_home"]

pooled_model = sm.GLM(stacked_last_season["team_xg"],
                      sm.add_constant(stacked_last_season[pooled_features]),
                      family=sm.families.Poisson()).fit()

print(pooled_model.summary().tables[1])
print("learned home advantage:", np.exp(pooled_model.params["is_home"]))

In [ ]:
pooled_predictions = pooled_model.predict(
    sm.add_constant(stacked_this_season[pooled_features], has_constant="add"))

for side_name, home_flag, separate_predictions, flat_value in [
        ("home", 1, p_h, flat_h), ("away", 0, p_a, flat_a)]:
    side_mask = stacked_this_season["is_home"] == home_flag
    actual_xg = stacked_this_season.loc[side_mask, "team_xg"]
    print(side_name,
          "| pooled:", round(mean_poisson_deviance(actual_xg, pooled_predictions[side_mask]), 3),
          "| separate:", round(mean_poisson_deviance(actual_xg, separate_predictions), 3),
          "| flat:", round(mean_poisson_deviance(actual_xg, [flat_value] * len(actual_xg)), 3))

In [ ]:
#### POISSON REGRESSION
pois_home, pois_away = fit_goal_models(train, FEATURES)
test, predictions = predict_fixtures(test, train, pois_home, pois_away, FEATURES, teams,
                                     output_path="predicted_fixtures_GW.csv")

## for walk forward
# train = train.sort_values(["season", "gameweek"])

In [ ]:
train["home_set"]  = train["home_weighted_set_play_strength"]  - train["away_weighted_set_play_defence"]
train["home_open"] = train["home_weighted_open_play_strength"] - train["away_weighted_open_play_defence"]
train["away_set"]  = train["away_weighted_set_play_strength"]  - train["home_weighted_set_play_defence"]
train["away_open"] = train["away_weighted_open_play_strength"] - train["home_weighted_open_play_defence"]

experiment_train = train.dropna(subset=["home_set", "home_open", "away_set", "away_open"])
tr = experiment_train[experiment_train["season"] == 0]
ho = experiment_train[experiment_train["season"] == 1]

def score(y_col, feats):
    m = sm.GLM(tr[y_col], sm.add_constant(tr[feats]), family=sm.families.Poisson()).fit()
    p = m.predict(sm.add_constant(ho[feats], has_constant="add"))
    return round(mean_poisson_deviance(ho[y_col], p), 4), m

for side_name, y, s, o in [("home", "home_non_penalty_xg", "home_set", "home_open"),
                           ("away", "away_non_penalty_xg", "away_set", "away_open")]:
    print(side_name,
          "elo-only", score(y, ["elo_context"])[0],
          "| set+elo", score(y, [s, "elo_context"])[0],
          "| set+open+elo", score(y, [s, o, "elo_context"])[0])

print("away xG mean  last season:", tr["away_non_penalty_xg"].mean(),
      "| this season:", ho["away_non_penalty_xg"].mean())
print("home xG mean  last season:", tr["home_non_penalty_xg"].mean(),
      "| this season:", ho["home_non_penalty_xg"].mean())

In [ ]:
# ---- FIXTURE SCORE from predictor (single GW) ----
team_fx = build_team_fixture_scores(test)
print(merged["player_id"].duplicated().sum())

In [ ]:
merged = attach_team_and_fixture_scores(merged, team_fixtures, team_fx)

In [ ]:
###LAST SEASON'S STATS
merged = attach_last_season_stats(merged, last_year_stats)

In [ ]:
merged = attach_minutes_factors(merged, LAST_GW)

In [ ]:
merged = attach_expected_return_signals(merged, LAST_GW)

In [ ]:
board = assemble_board(merged)

In [ ]:
all_boards = build_all_boards(config.LEAGUES)

In [ ]:
enriched = enrich_and_export(board, all_boards, LAST_GW, output_path="weekly_board_enriched.csv")
print_player_debug(merged)